# 第 14 章 ロジットとプロビット: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/14-logit-probit.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種
n = 300              # 標本の大きさ
link = "logit"       # データを作る誤差の分布（"logit" か "probit"）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## ロジット・プロビット・線形確率モデル（データを作った係数: 0.5・1・−0.8）

In [ ]:
import numpy as np
import pandas as pd

from lib.chapter14.discrete import (
    average_marginal_effects,
    draw_binary_sample,
    fit_binary,
    linear_probability,
)

sample = draw_binary_sample(n, link, np.random.default_rng(seed))
x = sample[["x1", "x2"]]
logit = fit_binary(sample["y"], x, "logit")
probit = fit_binary(sample["y"], x, "probit")
coefficients = pd.DataFrame(
    {"logit": logit.coefficients, "probit": probit.coefficients}
)
print(coefficients.round(3))
print(f"対数尤度: logit {logit.log_likelihood:.2f}、probit {probit.log_likelihood:.2f}")
print("平均限界効果")
print(pd.DataFrame({
    "logit": average_marginal_effects(logit, x),
    "probit": average_marginal_effects(probit, x),
    "線形確率": linear_probability(sample["y"], x).drop("const"),
}).round(3))

## 図: 確率の曲線

In [ ]:
from lib.chapter14.plot import plot_probability_curves

plot_probability_curves(sample["x1"].to_numpy(), sample["y"].to_numpy())

## 試してみよう

- `link` を `"probit"` にしても、ロジットとプロビットの平均限界効果は、
  ほぼ同じです（x1 が 0.275 と 0.276）。係数の大きさは違っても（ロジットは、
  プロビットの 1.7 倍ほど）、限界効果は同じです
- `n` を 3000 にすると、係数が真の値に近づきます（ロジットの x1 は 1.043、x2 は
  −0.791。潜在変数の係数は 1 と −0.8）
- 線形確率モデルの平均限界効果（x1 が 0.186）も、ロジット（0.184）に近い値です